In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt 
import plotly_express as px 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [2]:
data = yf.download("AAPL", start='2020-01-01', end='2026-04-17')

[*********************100%***********************]  1 of 1 completed


In [3]:
data.head()

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2020-01-02,72.400520,72.460784,71.156682,71.409785,135480400
2020-01-03,71.696648,72.455966,71.472469,71.629153,146322800
2020-01-06,72.267952,72.306521,70.568525,70.819223,118387200
2020-01-07,71.928024,72.533064,71.708665,72.277548,108872000
2020-01-08,73.085121,73.386438,71.631567,71.631567,132079200


In [4]:
data.columns = data.columns.droplevel(1)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1580 entries, 2020-01-02 to 2026-04-16
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   1580 non-null   float64
 1   High    1580 non-null   float64
 2   Low     1580 non-null   float64
 3   Open    1580 non-null   float64
 4   Volume  1580 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 74.1 KB


In [6]:
# Calculating Price Change
data['Price_Change'] = data['Close'].diff()
data['pct_change'] = data['Close'].pct_change() * 100
data.fillna(0,inplace=True)
data.head(5)

Price,Close,High,Low,Open,Volume,Price_Change,pct_change
Date,,,,,,,
2020-01-02,72.400520,72.460784,71.156682,71.409785,135480400,0.000000,0.000000
2020-01-03,71.696648,72.455966,71.472469,71.629153,146322800,-0.703873,-0.972193
2020-01-06,72.267952,72.306521,70.568525,70.819223,118387200,0.571304,0.796835
2020-01-07,71.928024,72.533064,71.708665,72.277548,108872000,-0.339928,-0.470371
2020-01-08,73.085121,73.386438,71.631567,71.631567,132079200,1.157097,1.608687


In [7]:
data.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Price_Change', 'pct_change'], dtype='object', name='Price')

In [8]:
# Seprating Gain and loss
data["Gain"] = data["Price_Change"].clip(lower=0)
data["Loss"] = -data["Price_Change"].clip(upper=0)

In [9]:
# Rolling Average
data['Avg_Gain'] = data['Gain'].rolling(5).mean()
data['Avg_Loss'] = data['Loss'].rolling(5).mean()

In [10]:
# Relative Strength
data['RS'] = data['Avg_Gain'] / data['Avg_Loss']

In [11]:
# RSI 
data['RSI'] = 100 - (100/(1 + data['RS']))

In [12]:
# Signal
data["Signal"] = "HOLD"
data.loc[(data["RSI"] < 30) & (data["RSI"].shift(1) >= 30), "Signal"] = "BUY"
data.loc[(data["RSI"] > 70) & (data["RSI"].shift(1) <= 70), "Signal"] = "SELL"

In [13]:
data.dropna(inplace=True)

In [14]:
data.head(21)

Price,Close,High,Low,Open,Volume,Price_Change,pct_change,Gain,Loss,Avg_Gain,Avg_Loss,RS,RSI,Signal
Date,,,,,,,,,,,,,,
2020-01-08,73.085121,73.386438,71.631567,71.631567,132079200,1.157097,1.608687,1.157097,-0.000000,0.345680,0.208760,1.655873,62.347602,HOLD
2020-01-09,74.637512,74.830352,73.810699,74.061390,170108400,1.552391,2.124086,1.552391,-0.000000,0.656158,0.208760,3.143122,75.863614,SELL
2020-01-10,74.806244,75.370316,74.304855,74.871333,140644800,0.168732,0.226068,0.168732,-0.000000,0.689905,0.067986,10.147817,91.029634,HOLD
2020-01-13,76.404396,76.430916,75.003874,75.121996,121532000,1.598152,2.136389,1.598152,-0.000000,0.895274,0.067986,13.168601,92.942140,HOLD
2020-01-14,75.372719,76.551476,75.249786,76.341760,161954400,-1.031677,-1.350285,0.000000,1.031677,0.895274,0.206335,4.338927,81.269643,HOLD
2020-01-15,75.049683,76.052467,74.618194,75.172622,121923600,-0.323036,-0.428585,0.000000,0.323036,0.663855,0.270943,2.450168,71.015900,HOLD
2020-01-16,75.989807,76.100697,75.230489,75.592070,108829200,0.940125,1.252670,0.940125,-0.000000,0.541402,0.270943,1.998215,66.646819,HOLD
2020-01-17,76.831100,76.833506,75.931967,76.238103,137816400,0.841293,1.107113,0.841293,-0.000000,0.675914,0.270943,2.494675,71.385037,SELL
2020-01-21,76.310410,76.900987,76.173007,76.459862,110843200,-0.520691,-0.677709,0.000000,0.520691,0.356284,0.375081,0.949885,48.714915,HOLD


In [15]:
data["Target"] = np.where(data["Close"].shift(-1) > data["Close"], "BUY", "SELL")

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
Y = le.fit_transform(data["Target"])

In [17]:
features = ["RSI", "Avg_Gain", "Avg_Loss", "Price_Change", "pct_change"]

X = data[features]
Y = data["Target"]

In [18]:
# Train test Split 
split = int(len(data)*0.8)

X_train = X[:split]
X_test = X[split:]

Y_train = Y[:split]
Y_test = Y[split:]

In [19]:
# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [27]:
# LogisticRegression 
model_lr = LogisticRegression()
model_lr.fit(X_train,Y_train)

# Prediction
y_pred_lr = model_lr.predict(X_test)

# Accuracy Check
print('Accutacy :', accuracy_score(Y_test,y_pred_lr))

result = pd.DataFrame({
    'Actual': Y_test,
    'Prediction': y_pred_lr
})
result.head()

Accutacy : 0.5158227848101266


,Actual,Prediction
Date,,
2025-01-13,SELL,SELL
2025-01-14,BUY,BUY
2025-01-15,SELL,BUY
2025-01-16,BUY,SELL
2025-01-17,SELL,BUY


In [32]:
# RandomForestClassifier
model_rf = RandomForestClassifier()
model_rf.fit(X_train,Y_train)

#Prediction
y_pred_rf = model_lr.predict(X_test)

#Accuracy check
print('accuracy_score :', accuracy_score(Y_test,y_pred_rf))

#Prediction check
result_rf = pd.DataFrame({
    'Actual':Y_test,
    'Predicted':y_pred_rf
})
result_rf.head()

accuracy_score : 0.5158227848101266


,Actual,Predicted
Date,,
2025-01-13,SELL,SELL
2025-01-14,BUY,BUY
2025-01-15,SELL,BUY
2025-01-16,BUY,SELL
2025-01-17,SELL,BUY


In [21]:
data.head()

Price,Close,High,Low,Open,Volume,Price_Change,pct_change,Gain,Loss,Avg_Gain,Avg_Loss,RS,RSI,Signal,Target
Date,,,,,,,,,,,,,,,
2020-01-08,73.085121,73.386438,71.631567,71.631567,132079200,1.157097,1.608687,1.157097,-0.000000,0.345680,0.208760,1.655873,62.347602,HOLD,BUY
2020-01-09,74.637512,74.830352,73.810699,74.061390,170108400,1.552391,2.124086,1.552391,-0.000000,0.656158,0.208760,3.143122,75.863614,SELL,BUY
2020-01-10,74.806244,75.370316,74.304855,74.871333,140644800,0.168732,0.226068,0.168732,-0.000000,0.689905,0.067986,10.147817,91.029634,HOLD,BUY
2020-01-13,76.404396,76.430916,75.003874,75.121996,121532000,1.598152,2.136389,1.598152,-0.000000,0.895274,0.067986,13.168601,92.942140,HOLD,SELL
2020-01-14,75.372719,76.551476,75.249786,76.341760,161954400,-1.031677,-1.350285,0.000000,1.031677,0.895274,0.206335,4.338927,81.269643,HOLD,SELL


In [23]:
X_test = pd.DataFrame(X_test, index=Y_test.index)

In [24]:
data.loc[X_test.index, 'ML_Signal'] = y_pred_lr

In [25]:
def final_signal(row):
    if row['Signal'] == 'BUY' and row['ML_Signal'] == 'BUY':
        return 'STRONG BUY'
    
    elif row['Signal'] == 'SELL' and row['ML_Signal'] == 'SELL':
        return 'STRONG SELL'
    
    elif row['ML_Signal'] == 'BUY':
        return 'WEAK BUY'
    
    elif row['ML_Signal'] == 'SELL':
        return 'WEAK SELL'
    
    else:
        return 'HOLD'
    
data['Final_Signal'] = data.apply(final_signal, axis = 1)